In [ ]:
%reload_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)
import pickle

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np

import numpyro
import numpyro.distributions as dist
from numpyro import handlers
from einops import repeat

import matplotlib.pyplot as plt
plt.rcParams.update({'text.usetex': False}) #comment out this line if you want to use LaTeX for the figure labels

from fpp.models.scd import dnds
from fpp.likelihoods.npll_jax import log_like_np
from fpp.utils.posterior import multi_corner

from fpp.models.np_model import NPModel
from fpp.utils.utils import jnp_trapezoid

# 1. Test `numpyro` with a small model
We first test the `numpyro` fitting pipeline to Fermi data using a small model. This will check that the code is working, and give us an rough estimate of the various template normalizations.

In [ ]:
# The key function to modify is the model() function, which defines the forward probabilistic model.
# We will reuse the same class as in our fiducial fit, NPModel.

class NPModelSmall(NPModel):

    def __init__(self):
        super().__init__(
            n_exp=1, # single exposure region
            l_max=0, # no spherical harmonic modulation of pibrem
            diffuse_names=["ModelO"], # only Model O for pibrem and ics for now
        )

    def model(self, data=None, beta=1.):

        mu = jnp.zeros_like(data)

        #=== diffuse: pibrem, ics ===
        mu += numpyro.sample("S_pib", dist.Uniform(1e-3, 14)) * self.pib[0] # Specifies a random variable S_pib and adding it to mu, the diffuse portion of the expected counts.
        mu += numpyro.sample("S_ics", dist.Uniform(1e-3, 14)) * self.ics[0]

        #=== diffuse: isotropic, fermi bubble, (resolved) point sources ===
        mu += numpyro.sample("S_iso", dist.Uniform(1e-3, 5.)) * self.temp_iso
        mu += numpyro.sample("S_bub", dist.Uniform(1e-3, 5.)) * self.temp_bub
        mu += numpyro.sample("S_psc", dist.Uniform(1e-3, 5.)) * self.temp_psc

        #=== diffuse: gce (=nfw) ===
        S_nfw = numpyro.sample("S_nfw", dist.Uniform(1e-5, 4.))
        temp_nfw_poiss = self.nfw_temp_gen.get_NFW2_template(gamma=0.9) # This gamma value can be inferred. For now we fix it.
        temp_nfw_poiss /= jnp.mean(temp_nfw_poiss[~self.nm])
        mu += S_nfw * temp_nfw_poiss

        #=== point source: disk ===
        Sps_dsk = numpyro.sample("Sps_dsk", dist.Uniform(1e-5, 4.))
        temp_dsk_ps = self.dsk_temp_gen.get_template(zs=0.5, C=3)
        temp_dsk_ps /= jnp.mean(temp_dsk_ps[~self.nm])

        npt_compressed = jnp.array([temp_dsk_ps])
        theta = []
        s_arr = jnp.logspace(-1., 2., 1000)
        n1 = numpyro.sample(f'n1_dsk', dist.Uniform(2.1, 8))
        n2 = numpyro.sample(f'n2_dsk', dist.Uniform(0.5, 2))
        n3 = numpyro.sample(f'n3_dsk', dist.Uniform(-8, -1))
        sb1 = numpyro.sample(f'sb1_dsk', dist.Uniform(5., 40.0))
        lambda_s = numpyro.sample(f'lambdas_dsk', dist.Uniform(0.1, 0.95))

        theta_tmp = jnp.array([1., n1, n2, n3, sb1, lambda_s * sb1])
        dnds_arr = dnds(s_arr, theta_tmp)
        A = Sps_dsk / jnp_trapezoid(s_arr * dnds_arr, s_arr)
        theta.append([A, n1, n2, n3, sb1, lambda_s * sb1])
        theta = jnp.array(theta)

        #=== point source: adjust with exposure regions ===
        # This can be ignored for now since we have only one exposure region.
        exp_lens = [len(self.expreg_indices[i]) for i in range(len(self.expreg_indices))]
        n_pad = exp_lens[0] - exp_lens[-1]

        expreg_indices = jnp.zeros_like(self.expreg_indices)
        expreg_indices = expreg_indices.at[:-1].set(self.expreg_indices[:-1])
        expreg_indices = expreg_indices.at[-1].set(jnp.pad(self.expreg_indices[-1], (0, n_pad)))

        log_like_np_exp_vmapped = jax.vmap(log_like_np, in_axes=(0, 0, 1, 0, None, None, None, None))

        mu_batch = mu[~self.mask_roi][jnp.array(expreg_indices)]
        npt_compressed_batch = npt_compressed[:, ~self.mask_roi][:, jnp.array(expreg_indices)]
        data_batch = data[~self.mask_roi][jnp.array(expreg_indices)]
        exposure_multiplier = self.exposure_means_list / self.exposure_mean

        theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
        theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
        theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
        theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])

        #=== compute likelihood ===
        with numpyro.plate("data", size=len(mu[~self.mask_roi]), dim=-1):

            log_like_exp = log_like_np_exp_vmapped(
                theta,
                mu_batch,
                npt_compressed_batch,
                data_batch,
                self.f_ary,
                self.df_rho_ary,
                self.k_max,
                len(expreg_indices[0]),
            )
            loglike = jnp.concatenate(log_like_exp)[:len(mu[~self.mask_roi])]

            with handlers.mask(mask=~jnp.logical_or(jnp.isinf(loglike), jnp.isnan(loglike))):
                with handlers.scale(scale=beta):
                    return numpyro.factor('log-likelihood', loglike)

In [ ]:
m = NPModelSmall()

In [ ]:
# SVI fit (~ 5 minutes on A100 GPU)
m.fit_svi(
    data=m.fermi_data, # Fermi data loaded by default, but fit_svi requires data to be passed in explicitly.
    guide='iaf', num_flows=4, hidden_dims=[64, 64], # SVI guide configuration: an (smaller) inverse autoregressive flow.
    lr=3e-4, n_steps=5000, # Adjust depending on convergence, 5000 steps should be sufficient.
)
samples_svi = m.get_svi_samples(num_samples=50000)

In [ ]:
# NUTS fit (~ 30 minutes on A100 GPU)
mcmc = m.run_nuts(
    data=m.data, # Similarly, data must be passed in explicitly for NUTS HMC as well.
    num_chains=4, num_warmup=500, # MCMC configuration: 4 chains with 500 warmup steps each.
    num_samples=2000, step_size=0.05, # 2000 * 4 = 8000 total samples.
)
samples_hmc = m.expand_samples(mcmc.get_samples())

In [ ]:
# saving
pickle.dump(samples_svi, open("svi_samples_small.pkl", "wb"))
pickle.dump(samples_hmc, open("hmc_samples_small.pkl", "wb"))

In [ ]:
# loading from a previous save
with open("svi_samples_small.pkl",'rb') as file:
    samples_svi=pickle.load(file)
    
with open("hmc_samples_small.pkl",'rb') as file:
    samples_hmc=pickle.load(file)

# 2. View posteriors

## 2.1 Full triangle plot

In [ ]:
plot_keys = ['S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_nfw', 'Sps_dsk', 'n1_dsk', 'n2_dsk', 'n3_dsk', 'sb1_dsk', 'lambdas_dsk']
latex_labels = [
    r'$S_\mathrm{pib}$', r'$S_\mathrm{ics}$', r'$S_\mathrm{iso}$', r'$S_\mathrm{bub}$', r'$S_\mathrm{nfw}$', r'$S^\mathrm{ps}_\mathrm{dsk}$',
    r'$n_1^\mathrm{dsk}$', r'$n_2^\mathrm{dsk}$', r'$n_3^\mathrm{dsk}$', r'$S_{b,1}^\mathrm{dsk}$', r'$\lambda_s^\mathrm{dsk}$'
]

multi_corner(
    {'hmc': samples_hmc, 'svi': samples_svi},
    plot_keys,
    labels=latex_labels,
    colors_dict={'hmc': 'k', 'svi': 'C0'},
    legend_dict={'hmc': r'$\text{HMC}$', 'svi': r'$\text{SVI}$'},
    legend_loc=(0.15, 0.94),
    label_kwargs={"fontsize": 30},
)

## 2.2 Visualize source count function posterior

In [ ]:
from fpp.utils.posterior import dnds_posterior

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

dnds_posterior(samples_svi, ['Sps_dsk', 'n1_dsk', 'n2_dsk', 'n3_dsk', 'sb1_dsk', 'lambdas_dsk'], ax=ax)

ax.set(xlim=(1, 30), ylim=(1e-6, 1e0))
ax.set(xlabel=r'$S$ [counts]', ylabel=r'$dN/dS$ [counts$^{-1}$]')
ax.set(title='Disk point source $dN/dS$ spatial mean (per pixel)');

# 3. Fiducial fit to *Fermi* data

Below is the code to get our fiducial fits. We recommend running this in a .py file and saving the results.

In [ ]:
m = NPModel()

In [ ]:
# SVI fit
m.fit_svi(
    data=m.fermi_data,
    guide='iaf', num_flows=5, hidden_dims=[128, 128],
    lr=3e-4, n_steps=10000,
)
samples_svi = np.array(m.get_svi_samples(num_samples=50000))

In [ ]:
# HMC fit
mcmc = m.run_nuts(
    data=m.fermi_data,
    num_chains=4, num_warmup=500,
    num_samples=7500, step_size=0.05, # 30000 samples for smooth posterior
)
samples_hmc = m.expand_samples(mcmc.get_samples())

In [ ]:
# saving
pickle.dump(samples_svi, open("svi_samples.pkl", "wb"))
pickle.dump(samples_hmc, open("hmc_samples.pkl", "wb"))